# Compare model-identity hardening without changing numerical inference


Preserve the original conformal prototype and compare its outputs to the final revision enforcing post-load canonical model payload identity. Code/model/calibration/artifact identities change; prediction sets, class scores and metric values must not change.

In [1]:
import json,hashlib
from pathlib import Path
from starplast import artifacts as A
from starplast.splits import read_split
roots=[Path('results/conformal_label_pilot_2026_10_08'),Path('results/conformal_label_pilot_2026_10_08_v2')]
loaded=[]
for root in roots:
    split,_=read_split(root/'split.json')
    model_spec=A.spec_from_dict(json.loads((root/'base_model/manifest.json').read_text())['spec'])
    model=A.read_artifact(root/'base_model',expected=model_spec,split=split)
    model.verify_contents()
    spec=A.spec_from_dict(json.loads((root/'held_out/manifest.json').read_text())['spec'])
    result=A.read_artifact(root/'held_out',expected=spec,split=split)
    result.verify_contents()
    metadata=json.loads((root/'manifest.json').read_text())
    for name,digest in metadata['input_sha256'].items():
        path=Path(name)
        if path.suffix=='.py':
            archived=root/'code'/path.name
            assert hashlib.sha256(archived.read_bytes()).hexdigest()==digest
    loaded.append((model,result,metadata))
assert loaded[0][0].payloads==loaded[1][0].payloads
for name in ('rows.json','class_scores.json','card.json','class_cards.json','baseline_cards.json'):
    assert loaded[0][1].payloads[name]==loaded[1][1].payloads[name],name
assert loaded[0][1].identity!=loaded[1][1].identity
assert loaded[0][0].identity!=loaded[1][0].identity
{'training_model_data_sets_scores_cards':'identical','model_and_artifact_ids':'correctly different','old_code_receipts':'exact archived copies','retained_test_entities':560}

{'training_model_data_sets_scores_cards': 'identical', 'model_and_artifact_ids': 'correctly different', 'old_code_receipts': 'exact archived copies', 'retained_test_entities': 560}